In [0]:
from pyspark.sql.functions import current_timestamp

sources = spark.sql("""SELECT * FROM facttransaction.lcf.source""")
# sources.display()
for source in sources.collect():
    # print(source.SourcePath)

    files = dbutils.fs.ls(source.SourcePath)
    # print(files)
    # file_name.rsplit(".", 1)[0]
    for df in files:
        table_name = df.name.rsplit(".", 1)[0]
        print(table_name)

        # dbutils.widgets.text("table_name", "")
        file_path = "/Volumes/facttransaction/bronze/sourcedata/" + source.SourceName + "/" + table_name + ".csv"
        # print(file_path)
        table_path = "facttransaction.bronze." + table_name #.split("_")[0] # use this if file name has date
        # print(table_path)

        df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(file_path)
        # df.withColumn("InsertDate", current_timestamp()).display()

        df.withColumn("InsertDate", current_timestamp()).write.mode("append").saveAsTable(table_path)


In [0]:
%sql
select * from facttransaction.bronze.tax order by 2, 1;

select concat_ws(",", Code, Cast(SourceSystemId as string), Rate, Description) from facttransaction.bronze.tax where InsertDate is not null and InsertDate < '2026-09-28 07:21:30';

